In [1]:
from transformers import AutoConfig, AutoTokenizer, BertTokenizer, BertForSequenceClassification, AutoModelForSequenceClassification, TrainingArguments, Trainer, pipeline
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.model_selection import GroupShuffleSplit, GroupKFold, KFold
from sklearn.metrics import f1_score, accuracy_score, roc_auc_score, classification_report
import pandas as pd
import re
import ast
import numpy as np
import seaborn as sns
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import wandb
from scipy.stats import entropy
import random
from transformers import set_seed as hf_set_seed
import torch

In [324]:
all_annotations = pd.read_csv("../../annotations/all_annotations.csv", skiprows=0)  # first row is data
all_annotations.columns = all_annotations.iloc[0]  # set first row as header
all_annotations = all_annotations[1:]

In [199]:
all_annotations

,NaN,orig_index,Filename,Period,Date,Speech #,Paragraph #,Speaker,Role,Gender,...,Interjection tokens,paragraph_token_count,interjection_token_count,Previous Interjections,partition,entropy,correct,merged_true,merged_pred,merged_correct
1,0.0,0,20_0002.xml,20,11.11.2021,3,7,Franziska Brantner,NaN,weiblich,...,"['So', 'ist', 'es']",0,3,[],unlabeled,0.02323954924941063,0.0,NaN,NaN,NaN
2,1.0,1,20_0002.xml,20,11.11.2021,4,1,Till Mansmann,NaN,männlich,...,['Wahnsinn'],0,1,[],unlabeled,0.0679970234632492,0.0,NaN,NaN,NaN
3,2.0,2,20_0002.xml,20,11.11.2021,4,3,Till Mansmann,NaN,männlich,...,"['Guter', 'Geschichtsunterricht']",0,2,"[('Beifall', 'FDP', None)]",unlabeled,0.24792788922786713,0.0,NaN,NaN,NaN
4,3.0,3,20_0002.xml,20,11.11.2021,4,4,Till Mansmann,NaN,männlich,...,['Richtig'],0,1,[],train,NaN,NaN,NaN,NaN,NaN
5,4.0,4,20_0002.xml,20,11.11.2021,4,4,Till Mansmann,NaN,männlich,...,"['Genau', 'Richtig']",0,2,"[('Zuruf', 'SPD', 'Richtig!')]",NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5779,5778.0,5778,20_0214.xml,20,18.03.2025,28,16,Sonja Eichwede,NaN,weiblich,...,"['Das', 'mit', 'den', 'Zinsen', 'haben', 'Sie'...",0,8,[],train,NaN,NaN,NaN,NaN,NaN
5780,5779.0,5779,20_0214.xml,20,18.03.2025,32,7,Jessica Rosenthal,NaN,weiblich,...,"['Dann', 'gehen', 'Sie', 'in', 'den', 'Landtag']",0,6,[],unlabeled,1.7606701850891113,0.0,NaN,NaN,NaN
5781,5780.0,5780,20_0214.xml,20,18.03.2025,37,8,Dennis Rohde,NaN,männlich,...,['Schleifen'],0,1,"[('Beifall', 'SPD', None), ('Zuruf', 'AfD', 'I...",unlabeled,0.6133553385734558,0.0,NaN,NaN,NaN
5782,5781.0,5781,20_0214.xml,20,18.03.2025,37,15,Dennis Rohde,NaN,männlich,...,"['Zu', 'Schulden']",0,2,[],unlabeled,1.81166410446167,0.0,NaN,NaN,NaN


In [3]:
top_1000 = (
    all_annotations
      .iloc[1:]
      .query('partition == "unlabeled" and flagged == "True"')
      .sort_values(by='entropy', ascending=False)
      .head(1000)
)

In [4]:
top_1000 = top_1000[top_1000["notes"] != "unclassifiable"]

In [ ]:
top_1000

,NaN,orig_index,Filename,Period,Date,Speech #,Paragraph #,Speaker,Role,Gender,...,Interjection tokens,paragraph_token_count,interjection_token_count,Previous Interjections,partition,entropy,correct,merged_true,merged_pred,merged_correct
542,541.0,541,20_0044.xml,20,23.06.2022,13,1,Andreas Mehltretter,NaN,männlich,...,"['Das', 'war', 'es', 'dann', 'auch']",0,5,[],unlabeled,2.564411163330078,0.0,NaN,NaN,NaN
3233,3232.0,3232,20_0146.xml,20,17.01.2024,9,4,Renate Künast,NaN,weiblich,...,"['Ganz', 'genau']",0,2,[],unlabeled,2.4306373596191406,0.0,NaN,NaN,NaN
3603,3602.0,3602,20_0157.xml,20,14.03.2024,3,16,Lena Werner,NaN,weiblich,...,"['Sehr', 'gut']",0,2,[],unlabeled,2.3580641746520996,0.0,NaN,NaN,NaN
3696,3695.0,3695,20_0160.xml,20,21.03.2024,8,1,Maja Wallstein,NaN,weiblich,...,"['Das', 'ist', 'bei', 'mir', 'im', 'Ruhrpott',...",0,7,[],unlabeled,2.3287832736968994,0.0,NaN,NaN,NaN
5090,5089.0,5089,20_0203.xml,20,05.12.2024,4,3,Sandra Bubendorfer-Licht,NaN,weiblich,...,"['und', 'ist', 'von', 'Ferdinand', 'Lassalle']",0,5,[],unlabeled,2.3263888359069824,0.0,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
603,602.0,602,20_0048.xml,20,08.07.2022,8,9,Martina Stamm-Fibich,NaN,weiblich,...,"['So', 'sieht', 'es', 'aus']",0,4,[],unlabeled,0.983078122138977,0.0,NaN,NaN,NaN
3056,3055.0,3055,20_0141.xml,20,30.11.2023,7,12,Dirk Wiese,NaN,männlich,...,"['So', 'ist', 'es']",0,3,"[('Beifall', 'SPD', None), ('Beifall', 'GRUENE...",unlabeled,0.9807325601577759,0.0,NaN,NaN,NaN
4252,4251.0,4251,20_0177.xml,20,26.06.2024,11,10,Natalie Pawlik,NaN,weiblich,...,"['Sehr', 'schön', 'Natalie']",0,3,"[('Beifall', 'SPD', None), ('Beifall', 'GRUENE...",unlabeled,0.9805208444595337,0.0,NaN,NaN,NaN
262,261.0,261,20_0026.xml,20,25.03.2022,5,4,Claudia Raffelhüschen,NaN,weiblich,...,['Ja'],0,1,[],unlabeled,0.9782618284225464,0.0,NaN,NaN,NaN


In [5]:
SEED = 42

# Python, NumPy, PyTorch seeds
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# HuggingFace helper
hf_set_seed(SEED)

In [ ]:
# Fine-tuning model one last time

In [6]:
# Load tokenizer
tokenizer = BertTokenizer.from_pretrained("hannahsteinbach/finetuned_parlBERT_phaseII")

# Load model
finetuned_parlbert = BertForSequenceClassification.from_pretrained("hannahsteinbach/finetuned_parlBERT_phaseII")

# Load config
config = AutoConfig.from_pretrained("hannahsteinbach/finetuned_parlBERT_phaseII")

# Get id2label and label2id
id2label = config.id2label
label2id = config.label2id
label_list = [label for idx, label in sorted(id2label.items())]

print("id2label:", id2label)
print("label2id:", label2id)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

id2label: {0: 'Macroeconomics', 1: 'Civil', 2: 'Health', 3: 'Agriculture', 4: 'Labor', 5: 'Education', 6: 'Environment', 7: 'Energy', 8: 'Immigration', 9: 'Transportation', 10: 'Law', 11: 'Social', 12: 'Housing', 13: 'Domestic', 14: 'Defense', 15: 'Technology', 16: 'Foreign', 17: 'International', 18: 'Government', 19: 'Public', 20: 'Culture'}
label2id: {'Agriculture': 3, 'Civil': 1, 'Culture': 20, 'Defense': 14, 'Domestic': 13, 'Education': 5, 'Energy': 7, 'Environment': 6, 'Foreign': 16, 'Government': 18, 'Health': 2, 'Housing': 12, 'Immigration': 8, 'International': 17, 'Labor': 4, 'Law': 10, 'Macroeconomics': 0, 'Public': 19, 'Social': 11, 'Technology': 15, 'Transportation': 9}


In [7]:
# Encode label
top_1000.loc[:, "Encoded Topic Label"] = top_1000["Topic Label"].map(label2id)

In [9]:
def safe_literal_eval(x):
    '''transform string of lists into lists'''
    if not x or pd.isna(x):
        return []
    try:
        return ast.literal_eval(x)
    except (ValueError, SyntaxError):
        return []

top_1000["Previous Paragraphs"] = top_1000["Previous Paragraphs"].apply(safe_literal_eval)

In [10]:
### preprocessing (umlaute)
def german_transliteration(text):
    def translit(s):
        return (s.replace("ä", "ae")
                 .replace("ö", "oe")
                 .replace("ü", "ue")
                 .replace("Ä", "Ae")
                 .replace("Ö", "Oe")
                 .replace("Ü", "Ue")
                 .replace("ß", "ss"))

    if isinstance(text, list):
        # If it's a list of strings
        return [translit(item) for item in text]
    elif isinstance(text, str):
        # If it's a single string
        return translit(text)
    else:
        # If it's something else (e.g., None), just return as is
        return text

columns_to_process = ["Paragraph", "Previous Paragraphs", "Supplementary Context", "Context", "Agenda Item"]

for col in columns_to_process:
    top_1000[f"{col}_encoded"] = top_1000[col].apply(german_transliteration)

In [ ]:
top_1000

,NaN,orig_index,Filename,Period,Date,Speech #,Paragraph #,Speaker,Role,Gender,...,correct,merged_true,merged_pred,merged_correct,Encoded Topic Label,Paragraph_encoded,Previous Paragraphs_encoded,Supplementary Context_encoded,Context_encoded,Agenda Item_encoded
542,541.0,541,20_0044.xml,20,23.06.2022,13,1,Andreas Mehltretter,NaN,männlich,...,0.0,NaN,NaN,NaN,0,Sehr geehrte Frau Praesidentin! Meine Damen un...,[],NaN,Beratung des Antrags der Fraktion der CDU/CSU,Teuerspirale beenden – Buergerinnen und Buerge...
3233,3232.0,3232,20_0146.xml,20,17.01.2024,9,4,Renate Künast,NaN,weiblich,...,0.0,NaN,NaN,NaN,18,"immer so oben in den Ueberschriften geredet, d...","[muss ich mal sagen., Sie haben ja wieder in s...",NaN,auf Verlangen der Fraktion der AfD,"Aktuelle Stunde. Landwirtschaft und Handwerk, ..."
3603,3602.0,3602,20_0157.xml,20,14.03.2024,3,16,Lena Werner,NaN,weiblich,...,0.0,NaN,NaN,NaN,15,"Ich freue mich sehr, dass wir diese Verordnung...",[Abschliessend noch ein Hinweis: Die Regelung ...,NaN,Zweite und dritte Beratung des von der Bundesr...,NaN
3696,3695.0,3695,20_0160.xml,20,21.03.2024,8,1,Maja Wallstein,NaN,weiblich,...,0.0,NaN,NaN,NaN,6,Hochgeschaetzte Frau Praesidentin! Werte Kolle...,[],NaN,Erste Beratung des von der Bundesregierung ein...,NaN
5090,5089.0,5089,20_0203.xml,20,05.12.2024,4,3,Sandra Bubendorfer-Licht,NaN,weiblich,...,0.0,NaN,NaN,NaN,14,Umso schmerzhafter ist jetzt der Sinneswandel....,[Herr Praesident! Meine Damen und Herren! Lieb...,NaN,Erste Beratung des von der Bundesregierung ein...,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
603,602.0,602,20_0048.xml,20,08.07.2022,8,9,Martina Stamm-Fibich,NaN,weiblich,...,0.0,NaN,NaN,NaN,2,Sie sehen also: Wir sind nicht untaetig. Wir t...,"[Aber ich bin mir sicher, dass oeffentliche In...",NaN,Beratung des Antrags der Fraktion der CDU/CSU,"Deutschland als Innovations-, Biotechnologie- ..."
3056,3055.0,3055,20_0141.xml,20,30.11.2023,7,12,Dirk Wiese,NaN,männlich,...,0.0,NaN,NaN,NaN,8,"Herr Throm, Sie haben es gerade ganz klar deut...",[Und wenn wir in den letzten 48 Stunden in den...,NaN,Erste Beratung des von der Bundesregierung ein...,NaN
4252,4251.0,4251,20_0177.xml,20,26.06.2024,11,10,Natalie Pawlik,NaN,weiblich,...,0.0,NaN,NaN,NaN,3,Vielen Dank fuer Ihre Aufmerksamkeit und einen...,"[Wir haben den Mindestlohn seinerzeit, 2015, e...",NaN,Beratung des Antrags der Abgeordneten Frank Ri...,Unsere Bauern retten – Ausnahmeregelung beim g...
262,261.0,261,20_0026.xml,20,25.03.2022,5,4,Claudia Raffelhüschen,NaN,weiblich,...,0.0,NaN,NaN,NaN,11,Fakt ist: Die Coronapandemie hat die globalen ...,[Sie und die folgenden Generationen sollen nic...,NaN,NaN,Wir kommen jetzt zu dem Geschaeftsbereich des ...


In [11]:
# -----------------------------
# 1. Tokenization function
# -----------------------------

def tokenize_multi_source(row, tokenizer, num_context=1, use_agenda_block=True, preprocess_agenda_block=True, max_length=256):
    """
    Tokenizes a single row with multiple sources: agenda, previous paragraphs, target paragraph.
    Keeps target at the end (segment_id=1), context/agenda at front (segment_id=0).

    Parameters:
        row: pd.Series, must contain 'Paragraph', 'Previous Paragraphs', 'Agenda Item', 'Context', 'Supplementary Context'
        tokenizer: HuggingFace tokenizer
        num_context: number of previous paragraphs to include
        use_agenda_block: include agenda/context/supplementary block if True
        max_length: max sequence length

    Returns:
        dict of input_ids, attention_mask, token_type_ids (all torch tensors)
    """
    # --- Previous paragraphs ---
    prev = row["Previous Paragraphs_encoded"]
    if isinstance(prev, str):
        prev = [prev]
    elif isinstance(prev, float) and pd.isna(prev):
        prev = []
    prev = prev[-num_context:] if num_context > 0 else []

    # --- Target paragraph ---
    target = str(row["Paragraph_encoded"]) if not pd.isna(row["Paragraph_encoded"]) else ""

    # --- Agenda block ---
    agenda_block = ""
    if use_agenda_block:
      if preprocess_agenda_block:
        agenda_block = " ".join(
            s for s in [row.get("Supplementary Context_preprocessed"), row.get("Context_preprocessed"), row.get("Agenda Item_preprocessed")]
            if pd.notna(s)
        )
      else:
        agenda_block = " ".join(
            s for s in [row.get("Supplementary Context_encoded"), row.get("Context_encoded"), row.get("Agenda Item_encoded")]
            if pd.notna(s)
        )

    # --- Combine sentences ---
    # Order: agenda/context first, previous paragraphs, target last
    all_sentences = []
    if agenda_block:
        all_sentences.append(agenda_block)
    all_sentences.extend(prev)
    all_sentences.append(target)

    # Join with [SEP]
    input_text = " [SEP] ".join(all_sentences)

    # Encode
    enc = tokenizer.encode_plus(
        input_text,
        add_special_tokens=True,
        truncation='only_first',  # truncate from front if too long
        padding='max_length',
        max_length=max_length,
        return_tensors='pt'
    )

    # --- token_type_ids ---
    # 0 for agenda/context/prev, 1 for target (last sentence)
    token_type_ids = []
    for idx, sentence in enumerate(all_sentences):
        length = len(tokenizer.tokenize(sentence))
        segment_id = 1 if idx == len(all_sentences) - 1 else 0
        token_type_ids.extend([segment_id] * length)
        token_type_ids.append(segment_id)  # for [SEP]

    # Add CLS at start
    token_type_ids = [0] + token_type_ids
    token_type_ids = (token_type_ids + [0]*max_length)[:max_length]

    return {
        "input_ids": enc["input_ids"].squeeze(),
        "attention_mask": enc["attention_mask"].squeeze(),
        "token_type_ids": torch.tensor(token_type_ids)
    }


def encode_dataframe(df, tokenizer, num_context, use_agenda_block, preprocess_agenda_block, max_length=256):
    all_input_ids = []
    all_attention = []
    all_token_types = []

    for _, row in df.iterrows():
        enc = tokenize_multi_source(
            row, tokenizer,
            num_context=num_context,
            use_agenda_block=use_agenda_block,
            preprocess_agenda_block=preprocess_agenda_block,
            max_length=max_length
        )
        all_input_ids.append(enc["input_ids"])
        all_attention.append(enc["attention_mask"])
        all_token_types.append(enc["token_type_ids"])

    return {
        "input_ids": torch.stack(all_input_ids),
        "attention_mask": torch.stack(all_attention),
        "token_type_ids": torch.stack(all_token_types),
    }


# -----------------------------
# 2. Dataset wrapper
# -----------------------------
class CustomDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

# -----------------------------
# 3. Metrics
# -----------------------------
def compute_metrics(eval_pred):
    """
    eval_pred: tuple (logits, labels)
        logits: raw model outputs (before softmax)
        labels: true class labels
    Returns:
        dict with f1_macro, f1_micro, f1_weighted, accuracy, auc_macro,
        avg_entropy, per_class_confidence
    """
    logits, labels = eval_pred
    probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()
    preds = np.argmax(probs, axis=-1)

    # --- Standard metrics ---
    f1_macro = f1_score(labels, preds, average="macro")
    f1_micro = f1_score(labels, preds, average="micro")
    f1_weighted = f1_score(labels, preds, average="weighted")
    accuracy = accuracy_score(labels, preds)

    # --- AUC ---
    try:
        auc_macro = roc_auc_score(labels, probs, multi_class="ovr", average="macro")
    except ValueError:
        auc_macro = float("nan")


    metrics = {
        "f1_macro": f1_macro,
        "f1_micro": f1_micro,
        "f1_weighted": f1_weighted,
        "accuracy": accuracy,
        "auc_macro": auc_macro,
    }

    return metrics


def model_init():
    model = BertForSequenceClassification.from_pretrained(
        "hannahsteinbach/finetuned_parlBERT_phaseI"
    )

    # Resize embeddings so BERT knows the new context tokens
    model.resize_token_embeddings(len(tokenizer))

    # Freeze first 8 layers (since we do not train the full model again)
    for param in model.bert.encoder.layer[:8].parameters():
        param.requires_grad = False

    return model


# Use previous best hyperparameters (parlbert_phaseII)
training_args = TrainingArguments(
    output_dir="./results_final",
    save_strategy="epoch",
    eval_strategy="epoch",
    learning_rate=2.672669365772627e-05,
    per_device_train_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.014810009452270324,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro"
)

In [12]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

In [ ]:
# encode train data

In [13]:
train_df = top_1000
train_labels = train_df["Encoded Topic Label"].to_list()

In [14]:
best_num_context=1
use_agenda_block=True
preprocess_agenda_block=False

train_enc = encode_dataframe(
    train_df, tokenizer,
    num_context=best_num_context,
    use_agenda_block=use_agenda_block,
    preprocess_agenda_block=preprocess_agenda_block,
    max_length=256
)

In [15]:
train_dataset = Dataset.from_dict({
    "input_ids": train_enc["input_ids"],
    "attention_mask": train_enc["attention_mask"],
    "token_type_ids": train_enc["token_type_ids"],
    "labels": torch.tensor(train_labels)
})

final_model = BertForSequenceClassification.from_pretrained(
    "hannahsteinbach/finetuned_parlBERT_phaseII"
)

# Freeze first 8 layers like before
for param in final_model.bert.encoder.layer[:8].parameters():
    param.requires_grad = False


# --- Trainer ---
training_args = TrainingArguments(
    output_dir="./results_final_final",
    save_strategy="no",
    learning_rate=2.672669365772627e-05,
    per_device_train_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.014810009452270324,
)

trainer = Trainer(
    model=final_model,
    args=training_args,
    train_dataset=train_dataset,
    tokenizer=tokenizer
)

# --- Train ---
trainer.train()

trainer.save_model("./finetuned_parlBERT_phaseIII")
tokenizer.save_pretrained("./finetuned_parlBERT_phaseIII")

/tmp/ipython-input-2492835540.py:27: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 3


wandb: You chose "Don't visualize my results"


Step,Training Loss


('./finetuned_parlBERT_phaseIII/tokenizer_config.json',
 './finetuned_parlBERT_phaseIII/special_tokens_map.json',
 './finetuned_parlBERT_phaseIII/vocab.txt',
 './finetuned_parlBERT_phaseIII/added_tokens.json')

In [ ]:
# Use new model to label unlabelled instances and obtain new threshold

In [74]:
rest_unlabeled = (
    all_annotations
      .iloc[1:]
      .query('partition == "unlabeled"')
      .drop(top_1000.index)
)

In [78]:
# Load tokenizer
tokenizer = BertTokenizer.from_pretrained("hannahsteinbach/finetuned_parlBERT_phaseIII")

# Load model
finetuned_parlbert = BertForSequenceClassification.from_pretrained("hannahsteinbach/finetuned_parlBERT_phaseIII")

# Load config
config = AutoConfig.from_pretrained("hannahsteinbach/finetuned_parlBERT_phaseIII")

# Get id2label and label2id
id2label = config.id2label
label2id = config.label2id
label_list = [label for idx, label in sorted(id2label.items())]

print("id2label:", id2label)
print("label2id:", label2id)


tokenizer_config.json: 0.00B [00:00, ?B/s]

C:\Users\hanna\miniconda3\envs\im2024\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\hanna\.cache\huggingface\hub\models--hannahsteinbach--finetuned_parlBERT_phaseIII. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

id2label: {0: 'Macroeconomics', 1: 'Civil', 2: 'Health', 3: 'Agriculture', 4: 'Labor', 5: 'Education', 6: 'Environment', 7: 'Energy', 8: 'Immigration', 9: 'Transportation', 10: 'Law', 11: 'Social', 12: 'Housing', 13: 'Domestic', 14: 'Defense', 15: 'Technology', 16: 'Foreign', 17: 'International', 18: 'Government', 19: 'Public', 20: 'Culture'}
label2id: {'Agriculture': 3, 'Civil': 1, 'Culture': 20, 'Defense': 14, 'Domestic': 13, 'Education': 5, 'Energy': 7, 'Environment': 6, 'Foreign': 16, 'Government': 18, 'Health': 2, 'Housing': 12, 'Immigration': 8, 'International': 17, 'Labor': 4, 'Law': 10, 'Macroeconomics': 0, 'Public': 19, 'Social': 11, 'Technology': 15, 'Transportation': 9}


In [ ]:
# preprocess

In [89]:
for col in columns_to_process:
    rest_unlabeled[f"{col}_encoded"] = rest_unlabeled[col].apply(german_transliteration)

In [90]:
unlabeled_enc = encode_dataframe(
    rest_unlabeled, tokenizer,
    num_context=best_num_context,
    use_agenda_block=use_agenda_block,
    preprocess_agenda_block=preprocess_agenda_block,
    max_length=256
)

In [91]:
unlabeled_dataset = Dataset.from_dict({
    "input_ids": unlabeled_enc["input_ids"],
    "attention_mask": unlabeled_enc["attention_mask"],
    "token_type_ids": unlabeled_enc["token_type_ids"],
})

def compute_entropy(probs):
    """Compute predictive entropy for each sample (raw values, no scaling)."""
    entropy = -torch.sum(probs * torch.log(probs + 1e-12), dim=-1)
    return entropy

final_model = BertForSequenceClassification.from_pretrained(
    "hannahsteinbach/finetuned_parlBERT_phaseIII"
)

# Freeze first 8 layers like before
for param in final_model.bert.encoder.layer[:8].parameters():
    param.requires_grad = False


# --- Trainer ---
training_args = TrainingArguments(
    output_dir="./results_final_final",
    save_strategy="no",
    learning_rate=2.672669365772627e-05,
    per_device_train_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.014810009452270324,
)

trainer = Trainer(
    model=final_model,
    args=training_args,
    train_dataset=unlabeled_dataset,
    tokenizer=tokenizer
)

# Get predictions (logits) from trainer
trainer.model.eval()
preds_final = trainer.predict(unlabeled_dataset)
logits_final = torch.tensor(preds_final.predictions, dtype=torch.float32)
probs_final = torch.softmax(logits_final, dim=-1)

preds_final = probs_final.argmax(dim=1).numpy()
entropy_final = compute_entropy(probs_final)

C:\Users\hanna\AppData\Local\Temp\ipykernel_1220\597629990.py:31: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [93]:
rest_unlabeled['entropy_phaseIII'] = entropy_final
rest_unlabeled['prediction_phaseIII'] =  [id2label[i] for i in preds_final]
rest_unlabeled['flagged_phaseIII'] = rest_unlabeled['entropy_phaseIII'] >= 0.346

In [97]:
rest_unlabeled['prediction_changed'] = (
    rest_unlabeled['prediction'] != rest_unlabeled['prediction_phaseIII']
)


In [100]:
# rest_unlabeled.to_csv("annotations_phaseIII.csv", index=False)

In [325]:
rest_unlabeled = pd.read_csv("../../annotations/annotations_phaseIII.csv", skiprows=0)

In [326]:
def fix_index(df):
    if "orig_index" in df.columns:
        df = df.set_index("orig_index")
    df.index = df.index.astype("Int64")
    return df

all_annotations = fix_index(all_annotations)
rest_unlabeled = fix_index(rest_unlabeled)

In [327]:
# ensure orig_index is index
if "orig_index" in all_annotations.columns:
    all_annotations = all_annotations.set_index("orig_index")

if "orig_index" in rest_unlabeled.columns:
    rest_unlabeled = rest_unlabeled.set_index("orig_index")

all_annotations = all_annotations[~all_annotations.index.duplicated(keep="last")]
rest_unlabeled = rest_unlabeled[~rest_unlabeled.index.duplicated(keep="last")]

all_annotations = all_annotations.drop(
    index=rest_unlabeled.index,
    errors="ignore"
)

all_annotations = pd.concat([all_annotations, rest_unlabeled])


In [328]:
all_annotations = all_annotations.loc[:, all_annotations.columns.notna()]

In [329]:
all_annotations['Period'] = (
    pd.to_numeric(all_annotations['Period'], errors='coerce')
    .astype('Int64')
)

In [331]:
# add topic label for paragraphs with several interjections (and only one paragraph has been annotated)
ignore_cols = [
    'Interjection Text',
    'Interjection Label',
    'Topic Label',
    'interjection_token_count',
    'Interjection tokens',
    'Previous Interjections',
    'Interjector',
    'Interjector Gender',
    'Interjector Party',
    'Interjection type',
    'Directed at (Person)',
    'Directed at (Party)',
    'orig_index',
    'nan',
    'partition',
    'entropy',
    'correct',
    'merged_true',
    'merged_pred',
    'merged_correct',
    'flagged',
    'prediction',
    'notes',
    'Topic Label_propagated',
    'Unnamed: 0',
    'prediction_phaseIII',
    'prediction_changed',
    'flagged_phaseIII',
    'entropy_phaseIII',
]

key_cols = [c for c in all_annotations.columns if c not in ignore_cols]

In [332]:
key_cols

['Filename',
 'Period',
 'Date',
 'Speech #',
 'Paragraph #',
 'Speaker',
 'Role',
 'Gender',
 'Party',
 'Previous Paragraphs',
 'Paragraph',
 'Agenda Item',
 'Context',
 'Supplementary Context',
 'Interjection',
 'Intervention',
 'Quote',
 'Verbal interjection',
 'Nonverbal interjection',
 'Paragraph tokens',
 'paragraph_token_count']

In [333]:
all_annotations['Topic Label'] = (
    all_annotations
    .groupby(key_cols, dropna=False)['Topic Label']
    .transform(lambda x: x.ffill().bfill())
)

C:\Users\hanna\AppData\Local\Temp\ipykernel_1220\1165997188.py:4: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .transform(lambda x: x.ffill().bfill())


In [334]:
all_annotations

,Filename,Period,Date,Speech #,Paragraph #,Speaker,Role,Gender,Party,Interjector Party,...,entropy,correct,merged_true,merged_pred,merged_correct,Unnamed: 0,prediction_phaseIII,prediction_changed,flagged_phaseIII,entropy_phaseIII
orig_index,,,,,,,,,,,,,,,,,,,,,
0,20_0002.xml,20,11.11.2021,3,7,Franziska Brantner,NaN,weiblich,GRUENE,SPD,...,0.02323954924941063,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,20_0002.xml,20,11.11.2021,4,4,Till Mansmann,NaN,männlich,FDP,SPD,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,20_0002.xml,20,11.11.2021,4,4,Till Mansmann,NaN,männlich,FDP,SPD,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,20_0002.xml,20,11.11.2021,4,5,Till Mansmann,NaN,männlich,FDP,FDP,...,1.4277269840240479,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,20_0002.xml,20,11.11.2021,9,6,Reinhard Houben,NaN,männlich,FDP,GRUENE,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5757,20_0213.xml,20,13.03.2025,5,16,Johannes Vogel,NaN,männlich,FDP,GRUENE,...,0.678934,0.0,NaN,NaN,NaN,5757.0,Macroeconomics,False,True,0.827598
5760,20_0213.xml,20,13.03.2025,4,17,Christian Dürr,NaN,männlich,FDP,GRUENE,...,0.785907,0.0,NaN,NaN,NaN,5760.0,Macroeconomics,False,True,0.499374
5764,20_0213.xml,20,13.03.2025,15,4,Achim Post,NaN,männlich,SPD,FDP,...,0.418373,0.0,NaN,NaN,NaN,5764.0,International,False,True,0.354835


In [335]:
all_annotations['manual_annotation'] = all_annotations['Topic Label'].notna()

In [ ]:
# fill annotations with predicted values and propagate predictions for same paragraphs

In [341]:
#  normalize types
for c in key_cols:
    if all_annotations[c].dtype.kind in ['i','f']:
        all_annotations[c] = pd.to_numeric(all_annotations[c], errors='coerce').astype('Int64')
    else:
        all_annotations[c] = all_annotations[c].astype(str).fillna('')

# fill missing Topic Label with predictions (i.e., not manually annotated)
all_annotations['Topic Label'] = all_annotations['Topic Label'].fillna(
    all_annotations['prediction_phaseIII']
)

all_annotations['Topic Label'] = (
    all_annotations
    .groupby(key_cols, dropna=False)['Topic Label']
    .transform(lambda x: x.ffill().bfill())
)


In [344]:
all_annotations.to_csv("../../annotations/final_labels_rq2.csv", index=False)